# Chapter 2: Classical Functionality: SCF, Localization, Stability

## What you'll learn

- Why canonical SCF orbitals are often unsuitable for quantum workflows — and how localization fixes this
- The built-in QDK localizers (`qdk_mp2_natural_orbitals`, `qdk_pipek_mezey`, `qdk_vvhv`) and the PySCF plugin localizer (Foster-Boys, Edmiston-Ruedenberg)
- How to interpret `get_summary()` to compare orbital character before and after localization
- What the stability checker does, when to use it, and how stability instability differs from convergence failure

## Why localize?

Canonical molecular orbitals from SCF are delocalized across the entire molecule: they're eigenstates of the Fock operator, optimized for energy, not for physical interpretability. For active space selection, delocalized orbitals obscure which electrons are strongly correlated. Localization rotates the orbital basis to produce spatially compact, chemically meaningful orbitals without changing the total energy or wavefunction.

In [ ]:
# Setup: imports and SCF
from pathlib import Path

import qdk_chemistry.plugins.pyscf

from qdk_chemistry.data import Structure
from qdk_chemistry.algorithms import available, create, print_settings
from qdk_chemistry.utils import Logger, compute_valence_space_parameters

Logger.set_global_level(Logger.LogLevel.off)

N2_XYZ = Path("../examples/data/stretched_n2.structure.xyz")
structure = Structure.from_xyz_file(N2_XYZ)

# Run SCF with cc-pvdz (same as Chapter 1 output)
scf_solver = create("scf_solver")
E_hf, wfn_hf = scf_solver.run(structure, charge=0, spin_multiplicity=1, basis_or_guess="cc-pvdz")
print(f"HF energy: {E_hf:.6f} Hartree")
print("\nCanonical orbital summary:")
print(wfn_hf.get_orbitals().get_summary())

## Part 1: Reducing to the valence space

Before localizing, we reduce to the **valence space**, the chemically relevant frontier orbitals. The N₂ cc-pvdz calculation produces 28 orbitals, most of which are high-energy virtuals that contribute negligibly to chemistry. Working in the valence space is a prerequisite for localization in this workflow.

`compute_valence_space_parameters()` is a utility that computes the number of valence electrons and orbitals for a given wavefunction. The `qdk_valence` active space selector then applies these to extract the valence subspace. Here, let's compute the valence space parameters for stretched N₂ and apply the valence selector. Take a look at the result.

In [ ]:
# Compute valence space parameters
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
print(f"Valence electrons: {num_val_e}")
print(f"Valence orbitals:  {num_val_o}")

# Select the valence subspace using qdk_valence
valence_selector = create(
    "active_space_selector",
    "qdk_valence",
    num_active_electrons=num_val_e,
    num_active_orbitals=num_val_o
)
wfn_valence = valence_selector.run(wfn_hf)

print("\nValence orbital summary:")
print(wfn_valence.get_orbitals().get_summary())

# Get the active space orbital indices for use in localization
valence_indices = wfn_valence.get_orbitals().get_active_space_indices()
print(f"\nActive space indices: {valence_indices}")

## Part 2: What localizers are available?

Now, let's use `available()` to list all registered orbital localizers. Take a minute to inspect the settings for each one.

In [ ]:
# List available orbital localizers
localizers = available("orbital_localizer")
print(f"Available orbital localizers: {localizers}")

In [ ]:
# Inspect localizer settings
for name in localizers:
    print(f"\n--- {name} ---")
    print_settings("orbital_localizer", name)

## Part 3: Applying MP2 natural orbital localization

`qdk_mp2_natural_orbitals` uses second-order Møller-Plesset perturbation theory to compute natural orbitals, which are the eigenstates of the correlated one-particle density matrix. These have fractional occupations that directly indicate correlation strength: occupations far from 0 or 2 flag strongly correlated orbitals.

The localizer's `run()` method takes the wavefunction and the active space index range (occupied start, virtual end). We will apply MP2 natural orbital localization to the valence space, and compare the orbital summary before and after.

In [ ]:
# Apply MP2 natural orbital localization
localizer_mp2 = create("orbital_localizer", "qdk_mp2_natural_orbitals")

# run() takes the wavefunction and the active space index bounds
wfn_mp2 = localizer_mp2.run(wfn_valence, *valence_indices)

print("Orbital summary after MP2 natural orbital localization:")
print(wfn_mp2.get_orbitals().get_summary())

## Part 4: Comparing localization methods

The QDK provides two additional localizers with different objectives:

- **`qdk_pipek_mezey`**: Maximizes orbital localization by minimizing the spread of each orbital over atomic centers (<a href="https://doi.org/10.1063/1.456588" target="_blank">Pipek & Mezey, 1989</a>). Produces orbitals resembling lone pairs and bonds. Takes the active space indices.
- **`qdk_vvhv`**: Valence-virtual hybrid localizer. Unlike the others, it requires **all virtual orbital indices** (from `n_alpha_electrons` to `num_molecular_orbitals-1`) — not just the active space. It localizes the entire virtual manifold, which is useful when downstream steps need well-localized virtual orbitals beyond the active space.

We will apply both localizers to `wfn_valence`, and compare all three orbital summaries side by side.

In [ ]:
# Apply Pipek-Mezey localization
localizer_pm = create("orbital_localizer", "qdk_pipek_mezey")
wfn_pm = localizer_pm.run(wfn_valence, *valence_indices)

print("Orbital summary after Pipek-Mezey localization:")
print(wfn_pm.get_orbitals().get_summary())

In [ ]:
# Apply VVHV localization
# VVHV differs from other localizers: it requires ALL virtual orbital indices
# (from n_alpha_electrons to num_molecular_orbitals-1), not just the active space.
localizer_vvhv = create("orbital_localizer", "qdk_vvhv")

num_alpha, num_beta = wfn_valence.get_total_num_electrons()
num_mos = wfn_valence.get_orbitals().get_num_molecular_orbitals()
vvhv_indices = (list(range(num_alpha, num_mos)), list(range(num_beta, num_mos)))

wfn_vvhv = localizer_vvhv.run(wfn_valence, *vvhv_indices)

print("Orbital summary after VVHV localization:")
print(wfn_vvhv.get_orbitals().get_summary())

In [ ]:
# Compare all localizer outputs
print("=" * 60)
print("CANONICAL (valence)")
print("=" * 60)
print(wfn_valence.get_orbitals().get_summary())

for label, wfn in [("MP2 natural orbitals", wfn_mp2), ("Pipek-Mezey", wfn_pm), ("VVHV", wfn_vvhv)]:
    print("=" * 60)
    print(label)
    print("=" * 60)
    print(wfn.get_orbitals().get_summary())

# What to look for in get_summary():
# - For MP2 natural orbitals: fractional occupations far from 0 or 2 flag strongly correlated orbitals
# - Active space size: a smaller active space with the same chemistry = a cheaper quantum calculation
# - Compare 'Active Orbitals' counts across methods to see how each partitions the orbital space

The summary tells you the shape of the orbital space after localization:

- **Active Orbitals**: how many orbitals are in the active space, which directly determines problem size downstream
- **Virtual Orbitals**: orbitals outside the active space; a non-zero count means some were excluded
- **Has inactive space**: whether frozen-core orbitals exist below the active space

For **MP2 natural orbitals** specifically, the key signal is occupation number: orbitals with occupations near 1 (far from 0 or 2) are strongly correlated. In stretched N₂ you should see at least two orbitals with occupations around 1: these are the bonding/antibonding σ pair being broken. That's the multi-reference character that motivates the active space workflow in Chapter 3.

For **Pipek-Mezey and Foster-Boys**, the active space size and orbital count reflect how compactly the electrons are described — a smaller active space with the same physics means a cheaper quantum circuit.

## Orbital visualization

The summary above captures orbital counts, but localization is fundamentally about **spatial character** — whether orbitals are spread across the whole molecule or concentrated on specific bonds. The cells below render orbital isosurfaces for the HOMO and LUMO before and after MP2 localization using the `MoleculeViewer` widget from the Microsoft QDK.

> **Note:** These cells require `qsharp_widgets` (`pip install qsharp_widgets`) and render in VS Code. 

### Canonical orbitals: before localization

In [ ]:
# Canonical orbitals: before localization
from qdk_chemistry.utils.cubegen import generate_cubefiles_from_orbitals
from qdk.widgets import MoleculeViewer

n_alpha, _ = wfn_valence.get_total_num_electrons()
show = [n_alpha - 1, n_alpha]  # HOMO and LUMO

cube_canonical = generate_cubefiles_from_orbitals(wfn_valence.get_orbitals(), indices=show)
MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_canonical, isoval=0.02)

In [ ]:
# Localized orbitals: after MP2 localization
cube_localized = generate_cubefiles_from_orbitals(wfn_mp2.get_orbitals(), indices=show)
MoleculeViewer(molecule_data=structure.to_xyz(), cube_data=cube_localized, isoval=0.02)

## Part 5: PySCF plugin localizer

The three built-in QDK localizers (`qdk_mp2_natural_orbitals`, `qdk_pipek_mezey`, `qdk_vvhv`) cover the most common workflows. Importing the PySCF plugin registers an additional `"pyscf_multi"` localizer that exposes PySCF's full localization suite via a `method` setting:

| `method` | Algorithm |
|---|---|
| `"pipek-mezey"` | Pipek-Mezey (default) |
| `"foster-boys"` | <a href="https://doi.org/10.1103/RevModPhys.32.300" target="_blank">Foster-Boys</a> |
| `"edmiston-ruedenberg"` | Edmiston-Ruedenberg |
| `"cholesky"` | Cholesky-based |

In the code cell below, we will apply Foster-Boys localization via the PySCF plugin and compare the orbital summary with the built-in Pipek-Mezey result from Part 4. To run the code, fill in the ```???``` first, and then answer the question below.

In [ ]:
# Apply Foster-Boys localization via PySCF plugin
# After import it registers 'pyscf_multi' as an additional orbital_localizer
print("Available localizers after PySCF import:", available("orbital_localizer"))

# Inspect the pyscf localizer's configurable methods
print()
print_settings("orbital_localizer", "pyscf_multi")

# Apply Foster-Boys via the PySCF localizer
localizer_fb = create("orbital_localizer", "pyscf_multi", method=???) # fill in this blank
wfn_fb = localizer_fb.run(wfn_valence, *valence_indices)

print("\nOrbital summary after Foster-Boys localization:")
print(wfn_fb.get_orbitals().get_summary())

print("\n--- Built-in Pipek-Mezey (for comparison) ---")
print(wfn_pm.get_orbitals().get_summary())

# Multiple Choice Question: ch2-foster-boys-tw78sa

Fill in the ```???``` blank in the Foster-Boys cell and run it. What is the number of active alpha orbitals you get from the orbital summary?

## Choices

A. 6
B. 12
C. 8
D. 16


## Part 6: SCF stability check

For some molecules, the Hartree-Fock wavefunction is not the true variational minimum because the SCF procedure can converge to a saddle point rather than a ground state. The stability checker tests whether the converged solution is stable against orbital rotations.

This is especially relevant for stretched bonds and open-shell systems, where restricted HF often yields an unstable solution. For stretched N₂, this check is important: broken-symmetry solutions may exist.

> **Stability instability vs convergence failure**: these are distinct problems.
> - *Convergence failure*: SCF never reaches a self-consistent solution — the iterations don't settle. Fix by tightening thresholds, changing the initial guess, or switching the SCF algorithm.
> - *Stability instability*: SCF converged, but to a saddle point — a lower-energy solution exists via an orbital rotation. The stability checker diagnoses this *after* a successful convergence.
> Stretched N₂ is a canonical example of the second case: SCF converges cleanly, but the restricted solution is not the ground state.

In the cell below, run the stability checker on the HF wavefunction. Is it stable? What does the checker return?

In [ ]:
# Inspect available stability checkers
print("Available stability checkers:", available("stability_checker"))
print()
print_settings("stability_checker", "pyscf")
# Key settings to note:
# - internal: tests stability within same wavefunction type (RHF stays RHF)
# - external: tests RHF → UHF instability (broken-symmetry solutions)
# - stability_tolerance: eigenvalue threshold — negative eigenvalue = unstable direction

In [ ]:
# Run the stability checker
stability_checker = create("stability_checker", "pyscf")
is_stable, stability_result = stability_checker.run(wfn_hf)

print(f"Overall stable:   {is_stable}")
print(f"Internal stable:  {stability_result.is_internal_stable()}")
print(f"External stable:  {stability_result.is_external_stable()}")
print(f"\nInternal eigenvalues: {stability_result.get_internal_eigenvalues()}")
print(f"External eigenvalues: {stability_result.get_external_eigenvalues()}")

# Interpreting the result:
# - Internal instability: a lower-energy solution exists within the same spin symmetry
# - External instability (RHF → UHF): a broken-symmetry unrestricted solution exists
#   This is common for stretched bonds — N₂ at 1.27 Å is expected to be externally unstable
#   because restricted HF cannot describe bond breaking correctly

# What to do when unstable:
# An external instability confirms that single-reference HF is an inadequate description.
# The correct response is NOT to re-run SCF, but to move to a multi-reference treatment —
# exactly what the active space + CASCI workflow in Chapters 3-4 provides.
# The localized MP2 natural orbital wavefunction (wfn_mp2) is the right starting point.
if not is_stable:
    print("\n→ Wavefunction is unstable. Proceed with multi-reference active space workflow (Chapter 3).")
else:
    print("\n→ Wavefunction is stable. Single-reference treatment may be adequate.")

# Multiple Choice Question: ch2-stab-anal-bouixk

 The stability checker returns ``is_stable=False, is_internal_stable=True, is_external_stable=False```. What is the correct interpretation and  
  next step?  

## Choices

A. SCF failed to converge — re-run with tighter thresholds 
B. The restricted HF solution is a saddle point; a lower-energy broken-symmetry UHF solution exists. Proceed to a multi-reference active space treatment.
C. The active space is too small — expand the number of active orbitals and re-run
D. The basis set is inadequate — switch to a larger basis


# Free Response Question: ch2-local-frq-26yvxg

All four get_summary() outputs in Part 4 (canonical valence, MP2, Pipek-Mezey, VVHV) show identical counts: Active Orbitals: α=8, β=8, Inactive: α=2, Virtual: α=18. A colleague concludes the localizers had no effect. What is wrong with this conclusion? What actually changed between the four wavefunctions, and why does that change matter for the active space selection step in the next chapter?


## Expected Answer

['localization changes which orbitals the active space selector "sees" as important, even when the total space size stays the same.']


## Summary

In this chapter you:
- Used `compute_valence_space_parameters()` and `qdk_valence` to reduce to the chemically relevant orbital subspace
- Applied three orbital localizers and compared their output via `get_summary()`
- Explored the PySCF plugin localizer (Foster-Boys, Edmiston-Ruedenberg) and ran the stability checker to test whether the HF wavefunction is a true energy minimum

The MP2-localized valence wavefunction (`wfn_mp2`) is the recommended input for active space selection in Chapter 3.

**Key pattern to remember:**
```python
num_val_e, num_val_o = compute_valence_space_parameters(wfn_hf, charge=0)
wfn_valence = create("active_space_selector", "qdk_valence",
                     num_active_electrons=num_val_e,
                     num_active_orbitals=num_val_o).run(wfn_hf)
indices = wfn_valence.get_orbitals().get_active_space_indices()
wfn_loc = create("orbital_localizer", "qdk_mp2_natural_orbitals").run(wfn_valence, *indices)
```